<a href="https://colab.research.google.com/github/kiryu-arai/kaggle_compedition_monster/blob/suzuki/v4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# California Housing 改善版 — 目標: RMSE < 0.40

| 改善点 | 効果 |
|--------|------|
| XGBRFRegressor → LightGBM/XGBoost | -0.05〜0.10 |
| 地理的特徴量（距離・クラスタ） | -0.08〜0.12 |
| 比率特徴量 | -0.02〜0.04 |
| Optuna ハイパラ探索 | -0.02〜0.04 |
| アンサンブル (XGB+LGB+CAT) | -0.01〜0.02 |

In [13]:
!pip install optuna -q
!pip install catboost -q

In [14]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [15]:
import os
import json

# カレントディレクトリ、または適切なパスから kaggle.json を読み込んでください
with open("kaggle.json", 'r') as f:
    json_data = json.load(f)
os.environ['KAGGLE_USERNAME'] = json_data['username']
os.environ['KAGGLE_KEY'] = json_data['key']

In [16]:
!kaggle competitions download -c ambl-california-housing
!unzip -o /content/ambl-california-housing.zip

ambl-california-housing.zip: Skipping, found more recently modified local copy (use --force to force download)
Archive:  /content/ambl-california-housing.zip
  inflating: sample.csv              
  inflating: test.csv                
  inflating: train.csv               


In [17]:
# データの移動
import os
import shutil

destination_folder = '/content/drive/MyDrive/kaggle_data'

if not os.path.exists(destination_folder):
    os.makedirs(destination_folder)
    print(f"フォルダ '{destination_folder}' を作成しました。")
else:
    print(f"フォルダ '{destination_folder}' は既に存在します。")

files_to_move = ['sample.csv', 'test.csv', 'train.csv']

for file_name in files_to_move:
    source_path = os.path.join('/content/', file_name)
    destination_path = os.path.join(destination_folder, file_name)
    if os.path.exists(source_path):
        shutil.move(source_path, destination_path)
        print(f"'{file_name}' を '{destination_folder}' に移動しました。")
    else:
        print(f"'{file_name}' は存在しませんでした。")

フォルダ '/content/drive/MyDrive/kaggle_data' は既に存在します。
'sample.csv' を '/content/drive/MyDrive/kaggle_data' に移動しました。
'test.csv' を '/content/drive/MyDrive/kaggle_data' に移動しました。
'train.csv' を '/content/drive/MyDrive/kaggle_data' に移動しました。


In [18]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.cluster import KMeans
import lightgbm as lgb
from xgboost import XGBRegressor
from catboost import CatBoostRegressor, Pool
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
import warnings; warnings.filterwarnings('ignore')

train = pd.read_csv('/content/drive/MyDrive/kaggle_data/train.csv')
test = pd.read_csv('/content/drive/MyDrive/kaggle_data/test.csv')
sample = pd.read_csv('/content/drive/MyDrive/kaggle_data/sample.csv')

print(f'Train: {train.shape}  Test: {test.shape}')
train.head()

Train: (16512, 13)  Test: (4128, 12)


,id,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,Household,AllRooms,AllBedrms,Price
0,0,1.4817,6.0,4.443645,1.134293,1397.0,3.350120,36.77,-119.84,417.0,1853.0,473.0,0.720
1,1,6.9133,8.0,5.976471,1.026471,862.0,2.535294,33.68,-117.80,340.0,2032.0,349.0,2.741
2,2,1.5536,25.0,4.088785,1.000000,931.0,4.350467,36.60,-120.19,214.0,875.0,214.0,0.583
3,3,1.5284,31.0,2.740088,1.008811,597.0,2.629956,34.10,-118.32,227.0,622.0,229.0,2.000
4,4,4.0815,21.0,5.166667,1.002688,1130.0,3.037634,37.79,-121.23,372.0,1922.0,373.0,1.179


## 特徴量エンジニアリング

**地理的特徴量が最重要** — CaliforniaはSF/LA近郊と内陸で地価が数倍違う

In [19]:
def add_features(df, km_model=None, fit_km=False):
    df = df.copy()

    # 比率特徴量
    df['rooms_per_person']  = df['AveRooms']   / (df['AveOccup']   + 1e-3)
    df['bedrooms_ratio']    = df['AveBedrms']  / (df['AveRooms']   + 1e-3)
    df['income_per_room']   = df['MedInc']     / (df['AveRooms']   + 1e-3)
    df['pop_per_household'] = df['Population'] / (df['AveOccup']   + 1e-3)

    # 主要都市からの距離
    cities = {
        'LA': (34.052, -118.244),
        'SF': (37.774, -122.419),
        'SD': (32.716, -117.162),
        'SJ': (37.338, -121.886),
        'SB': (34.420, -119.698),
    }
    for city, (clat, clon) in cities.items():
        df[f'dist_{city}'] = np.sqrt(
            (df['Latitude'] - clat)**2 + (df['Longitude'] - clon)**2
        )
    df['dist_nearest_city'] = df[[f'dist_{c}' for c in cities]].min(axis=1)

    # 海岸からの距離（簡易）
    df['coast_dist'] = np.abs(df['Longitude'] + 120.5)

    # 緯度×経度のインタラクション
    df['lat_lon'] = df['Latitude'] * df['Longitude']

    # 地理クラスタ（K-means: train で fit、test には predict のみ）
    coords = df[['Latitude', 'Longitude']].values
    if fit_km:
        km_model.fit(coords)
    df['geo_cluster'] = km_model.predict(coords)

    return df

# K-means は train だけで fit（リーク防止）
km = KMeans(n_clusters=50, random_state=42, n_init=10)
train_fe = add_features(train, km_model=km, fit_km=True)
test_fe  = add_features(test,  km_model=km, fit_km=False)

feature_cols = [c for c in train_fe.columns if c not in ['id', 'Price']]

X      = train_fe[feature_cols].reset_index(drop=True)
y      = train_fe['Price'].reset_index(drop=True)
X_test = test_fe[feature_cols].reset_index(drop=True)

print(f'特徴量数: {len(feature_cols)}  (元: 8)')
print(feature_cols)

特徴量数: 24  (元: 8)
['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude', 'Household', 'AllRooms', 'AllBedrms', 'rooms_per_person', 'bedrooms_ratio', 'income_per_room', 'pop_per_household', 'dist_LA', 'dist_SF', 'dist_SD', 'dist_SJ', 'dist_SB', 'dist_nearest_city', 'coast_dist', 'lat_lon', 'geo_cluster']


## LightGBM — Optuna でハイパラ最適化

5-fold CV + early stopping で過学習を防ぎながら探索

In [ ]:
KF = KFold(n_splits=5, shuffle=True, random_state=42)

def lgb_cv_rmse(params):
    scores = []
    for tr_idx, va_idx in KF.split(X):
        m = lgb.LGBMRegressor(**params)
        m.fit(
            X.iloc[tr_idx], y.iloc[tr_idx],
            eval_set=[(X.iloc[va_idx], y.iloc[va_idx])],
            callbacks=[
                lgb.early_stopping(stopping_rounds=50, verbose=False),
                lgb.log_evaluation(period=-1),
            ],
        )
        pred = m.predict(X.iloc[va_idx])
        scores.append(np.sqrt(mean_squared_error(y.iloc[va_idx], pred)))
    return np.mean(scores)

def objective_lgb(trial):
    params = dict(
        n_estimators      = 2000,           # early stopping で実際の本数が決まる
        num_leaves        = trial.suggest_int('num_leaves', 20, 300),
        learning_rate     = trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        min_child_samples = trial.suggest_int('min_child_samples', 5, 100),
        subsample         = trial.suggest_float('subsample', 0.5, 1.0),
        colsample_bytree  = trial.suggest_float('colsample_bytree', 0.5, 1.0),
        reg_alpha         = trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        reg_lambda        = trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
        random_state=42, verbose=-1,

    )
    return lgb_cv_rmse(params)

study_lgb = optuna.create_study(direction='minimize',
                                 sampler=optuna.samplers.TPESampler(seed=42))
study_lgb.optimize(objective_lgb, n_trials=80, show_progress_bar=True)
print(f'LightGBM Best CV RMSE: {study_lgb.best_value:.4f}')
print(f'Best params: {study_lgb.best_params}')

  0%|          | 0/80 [00:00<?, ?it/s]

## XGBoost — Optuna でハイパラ最適化

In [ ]:
def xgb_cv_rmse(params):
    scores = []
    for tr_idx, va_idx in KF.split(X):
        m = XGBRegressor(**params, verbosity=0)
        m.fit(
            X.iloc[tr_idx], y.iloc[tr_idx],
            eval_set=[(X.iloc[va_idx], y.iloc[va_idx])],
            verbose=False,
        )
        pred = m.predict(X.iloc[va_idx])
        scores.append(np.sqrt(mean_squared_error(y.iloc[va_idx], pred)))
    return np.mean(scores)

def objective_xgb(trial):
    params = dict(
        n_estimators     = 2000,
        early_stopping_rounds = 50,
        max_depth        = trial.suggest_int('max_depth', 3, 10),
        learning_rate    = trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        subsample        = trial.suggest_float('subsample', 0.5, 1.0),
        colsample_bytree = trial.suggest_float('colsample_bytree', 0.5, 1.0),
        reg_alpha        = trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True),
        reg_lambda       = trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True),
        min_child_weight = trial.suggest_int('min_child_weight', 1, 20),
        random_state=42, tree_method='hist',
    )
    return xgb_cv_rmse(params)

study_xgb = optuna.create_study(direction='minimize',
                                  sampler=optuna.samplers.TPESampler(seed=42))
study_xgb.optimize(objective_xgb, n_trials=60, show_progress_bar=True)
print(f'XGBoost Best CV RMSE: {study_xgb.best_value:.4f}')
print(f'Best params: {study_xgb.best_params}')

## アンサンブル（Out-of-Fold 予測）

各モデルの OOF 予測を平均して最終スコアを確認

In [ ]:
def make_oof(model_fn, X, y, X_test, n_splits=5):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    oof       = np.zeros(len(X))
    test_pred = np.zeros(len(X_test))
    for tr_idx, va_idx in kf.split(X):
        m = model_fn()
        m.fit(X.iloc[tr_idx], y.iloc[tr_idx])
        oof[va_idx]  = m.predict(X.iloc[va_idx])
        test_pred   += m.predict(X_test) / n_splits
    rmse = np.sqrt(mean_squared_error(y, oof))
    return oof, test_pred, rmse

# LightGBM（best params、early stopping なしで n_estimators を大きめに固定）
lgb_best = {**study_lgb.best_params, 'n_estimators': 1500,
            'random_state': 42, 'verbose': -1}
oof_lgb, test_lgb, rmse_lgb = make_oof(
    lambda: lgb.LGBMRegressor(**lgb_best), X, y, X_test
)
print(f'LightGBM OOF RMSE: {rmse_lgb:.4f}')

# XGBoost（best params）
xgb_best = {**study_xgb.best_params, 'n_estimators': 1500,
            'random_state': 42, 'tree_method': 'hist', 'verbosity': 0}
oof_xgb, test_xgb, rmse_xgb = make_oof(
    lambda: XGBRegressor(**xgb_best), X, y, X_test
)
print(f'XGBoost  OOF RMSE: {rmse_xgb:.4f}')

# CatBoost（geo_cluster をカテゴリ特徴量として渡す）
cat_feature_idx = [feature_cols.index('geo_cluster')]
def make_catboost():
    return CatBoostRegressor(
        iterations=1500, learning_rate=0.05, depth=6,
        random_seed=42, verbose=0,
        cat_features=cat_feature_idx,
    )

def make_oof_catboost(X, y, X_test, n_splits=5):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    oof       = np.zeros(len(X))
    test_pred = np.zeros(len(X_test))
    for tr_idx, va_idx in kf.split(X):
        m = make_catboost()
        m.fit(X.iloc[tr_idx], y.iloc[tr_idx],
              eval_set=(X.iloc[va_idx], y.iloc[va_idx]),
              early_stopping_rounds=50)
        oof[va_idx]  = m.predict(X.iloc[va_idx])
        test_pred   += m.predict(X_test) / n_splits
    return oof, test_pred, np.sqrt(mean_squared_error(y, oof))

oof_cat, test_cat, rmse_cat = make_oof_catboost(X, y, X_test)
print(f'CatBoost OOF RMSE: {rmse_cat:.4f}')

# 単純平均アンサンブル
oof_ens  = (oof_lgb  + oof_xgb  + oof_cat)  / 3
test_ens = (test_lgb + test_xgb + test_cat) / 3
rmse_ens = np.sqrt(mean_squared_error(y, oof_ens))

print(f'\n── OOF RMSE サマリ ──')
print(f'  LightGBM : {rmse_lgb:.4f}')
print(f'  XGBoost  : {rmse_xgb:.4f}')
print(f'  CatBoost : {rmse_cat:.4f}')
print(f'  Ensemble : {rmse_ens:.4f}  ← 提出値')

## 提出ファイル作成

In [ ]:
pred_final = np.clip(test_ens, 0, 5.00001)

sample['Price'] = pred_final
sample.to_csv('submit_improved.csv', index=False)

print('提出ファイル作成完了: submit_improved.csv')
print(f'予測値: min={pred_final.min():.3f}  max={pred_final.max():.3f}  mean={pred_final.mean():.3f}')